# Build 50-Hour Sports Audio Separator Dataset on Kaggle

This notebook builds a synthetic training dataset from LibriSpeech + ESC-50.

**Setup:**
1. Add these Kaggle datasets as inputs:
   - `librispeech` (search "LibriSpeech")
   - `esc-50` (search "ESC-50")
2. Run all cells
3. Download `dataset_out_50h/` from output files

**Output:** 36,000 examples (50 hours) in `/kaggle/working/dataset_out_50h/`

In [ ]:
import os
import csv
import json
import random
import subprocess
from pathlib import Path
from collections import defaultdict

# Kaggle paths
LIBRISPEECH_ROOT = "/kaggle/input/datasets/victorling/librispeech-clean/LibriSpeech/train-clean-100"
ESC50_ROOT = "/kaggle/input/datasets/niranjankn/esc50/ESC-50-master"
OUTPUT_DIR = "/kaggle/working/dataset_out_50h"

# Config
EXAMPLES = 36000
DURATION_SECONDS = 5.0
SPEECH_HOURS_CAP = 50.0
ESC50_CLASSES = [
    "airplane", "car_horn", "church_bells", "clapping", "engine",
    "fireworks", "footsteps", "helicopter", "laughing", "rain",
    "siren", "thunderstorm", "train", "wind"
]
SNR_MIN, SNR_MAX = 0.0, 12.0
CODEC = "flac"

print(f"LibriSpeech root: {LIBRISPEECH_ROOT}")
print(f"ESC-50 root: {ESC50_ROOT}")
print(f"Output: {OUTPUT_DIR}")
print(f"Target: {EXAMPLES} examples ({DURATION_SECONDS}s each)")
print()

### Collect LibriSpeech files


In [ ]:
print("Scanning LibriSpeech for FLAC files...")
librispeech_root = Path(LIBRISPEECH_ROOT)
speech_files = []
speech_hours = 0.0

for flac_path in sorted(librispeech_root.glob("**/*.flac")):
    if speech_hours >= SPEECH_HOURS_CAP:
        print(f"Reached {SPEECH_HOURS_CAP} hour cap.")
        break
    
    try:
        result = subprocess.run(
            ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "default=noprint_wrappers=1:nokey=1:nokey=1", str(flac_path)],
            capture_output=True, text=True, timeout=5
        )
        duration = float(result.stdout.strip())
        speech_files.append((str(flac_path), duration))
        speech_hours += duration / 3600.0
    except:
        pass

print(f"Collected {len(speech_files)} speech files (~{speech_hours:.1f} hours)")
print()

In [ ]:
# Collect ESC-50 files
print(f"Scanning ESC-50 for {len(ESC50_CLASSES)} classes...")
esc50_root = Path(ESC50_ROOT)
esc50_meta_path = esc50_root / "meta" / "esc50.csv"

esc50_files = {}
with open(esc50_meta_path, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        target_name = row.get("target", "").strip()
        if target_name in ESC50_CLASSES:
            filename = row.get("filename", "").strip()
            if filename:
                esc50_files[filename] = target_name

noise_files = []
for filename in esc50_files.keys():
    audio_path = esc50_root / "audio" / filename
    if audio_path.exists():
        noise_files.append(str(audio_path))

print(f"Collected {len(noise_files)} ESC-50 files from {len(set(esc50_files.values()))} classes")
print(f"Classes: {', '.join(sorted(set(esc50_files.values())))}")
print()

### Prepare output directories


In [ ]:
output_path = Path(OUTPUT_DIR)
output_path.mkdir(parents=True, exist_ok=True)

splits = {"train": 0.9, "val": 0.05, "test": 0.05}
for split in splits:
    for subdir in ["mixtures", "speech", "noise"]:
        (output_path / split / subdir).mkdir(parents=True, exist_ok=True)

metadata_rows = []
print("Generating examples...")
print()

### Generate examples


In [ ]:
def render_segment(audio_path, duration_sec, output_path):
    """Render audio segment to exact duration."""
    cmd = [
        "ffmpeg", "-i", audio_path,
        "-af", f"atrim=0:{duration_sec},apad=pad_len={int(16000 * duration_sec)}",
        "-ar", "16000", "-ac", "1", "-f", CODEC,
        "-y", output_path
    ]
    subprocess.run(cmd, capture_output=True, timeout=30)

def measure_mean_volume_db(audio_path):
    """Measure mean volume in dB."""
    cmd = [
        "ffmpeg", "-i", audio_path,
        "-af", "volumedetect", "-f", "null", "-"
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
    for line in result.stderr.split('\n'):
        if "mean_volume" in line:
            try:
                return float(line.split()[-2])
            except:
                return -20.0
    return -20.0

def make_example(example_id, speech_path, noise_path, output_dir, split):
    """Create mixed audio example."""
    stem_name = f"{split}_{example_id:04d}"
    
    speech_out = output_dir / split / "speech" / f"{stem_name}.{CODEC}"
    noise_out = output_dir / split / "noise" / f"{stem_name}.{CODEC}"
    mixture_out = output_dir / split / "mixtures" / f"{stem_name}.{CODEC}"
    
    # Render segments
    render_segment(speech_path, DURATION_SECONDS, str(speech_out))
    render_segment(noise_path, DURATION_SECONDS, str(noise_out))
    
    # Measure loudness
    speech_vol = measure_mean_volume_db(str(speech_out))
    noise_vol = measure_mean_volume_db(str(noise_out))
    
    # Compute gains
    snr = random.uniform(SNR_MIN, SNR_MAX)
    speech_gain_db = 0.0
    noise_gain_db = speech_vol - noise_vol - snr
    
    # Mix
    speech_gain_lin = 10 ** (speech_gain_db / 20.0)
    noise_gain_lin = 10 ** (noise_gain_db / 20.0)
    
    mix_cmd = [
        "ffmpeg", "-i", str(speech_out), "-i", str(noise_out),
        "-filter_complex", f"[0]volume={speech_gain_lin}[s];[1]volume={noise_gain_lin}[n];[s][n]amix=inputs=2:duration=first[out]",
        "-map", "[out]", "-ar", "16000", "-ac", "1", "-f", CODEC,
        "-y", str(mixture_out)
    ]
    subprocess.run(mix_cmd, capture_output=True, timeout=30)
    
    return {
        "example_id": stem_name,
        "split": split,
        "mixture": str(mixture_out.relative_to(output_dir)),
        "speech": str(speech_out.relative_to(output_dir)),
        "noise": str(noise_out.relative_to(output_dir)),
        "speech_source": speech_path,
        "noise_source": noise_path,
        "speech_gain_db": f"{speech_gain_db:.2f}",
        "noise_gain_db": f"{noise_gain_db:.2f}",
        "snr_db": f"{snr:.2f}",
    }

print("Building examples...")
example_id = 0

for idx in range(EXAMPLES):
    speech_path, _ = random.choice(speech_files)
    noise_path = random.choice(noise_files)
    
    # Assign split
    rand = random.random()
    if rand < 0.9:
        split = "train"
    elif rand < 0.95:
        split = "val"
    else:
        split = "test"
    
    try:
        metadata = make_example(example_id, speech_path, noise_path, output_path, split)
        metadata_rows.append(metadata)
        example_id += 1
    except Exception as e:
        print(f"Error on example {idx}: {e}")
        continue
    
    if (idx + 1) % 500 == 0:
        print(f"Built {idx + 1}/{EXAMPLES} examples")
        print()

print(f"Built {len(metadata_rows)} examples")
print()

### Write metadata CSV

In [ ]:
metadata_path = output_path / "metadata.csv"
with open(metadata_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=metadata_rows[0].keys())
    writer.writeheader()
    writer.writerows(metadata_rows)

print(f"Wrote metadata.csv with {len(metadata_rows)} rows")
print()

### Summary


In [ ]:
split_counts = defaultdict(int)
for row in metadata_rows:
    split_counts[row["split"]] += 1

total_audio_hours = (len(metadata_rows) * DURATION_SECONDS) / 3600.0

summary = {
    "output": OUTPUT_DIR,
    "examples": len(metadata_rows),
    "duration_seconds": DURATION_SECONDS,
    "speech_hours_cap": SPEECH_HOURS_CAP,
    "approx_mixture_hours": round(total_audio_hours, 2),
    "splits": dict(split_counts),
    "speech_source_count": len(speech_files),
    "noise_source_count": len(noise_files),
    "esc50_classes": ESC50_CLASSES,
    "codec": CODEC,
}

print(json.dumps(summary, indent=2))